In [4]:
import pandas as pd
from pprint import pprint

In [5]:
pd.set_option("display.max_columns", False)

In [ ]:
df = pd.read_parquet("./reference_answers.parquet")

# print(df.columns)
row = df.iloc[3]
# pprint(row['original_reference_response_raw_response'])
# pprint(row['original_question_prompt'])
df.iloc[0:10]
row['original_question']


'This is a male employee aged 18-30, single, with a college or below level of education. He works in the Sales department, holds an entry position, earns a low (<$3k) monthly salary. He has been at this company for 3-5 years - established. He does not work overtime, no travel, commutes a near (1-9 miles) distance.'

In [54]:
# Looking at the reasoning of the qwen model to see how it compares
df.columns.to_list()

['original_dataset',
 'original_question',
 'original_question_prompt',
 'original_question_idx',
 'original_ground_truth',
 'original_answer_first',
 'original_description',
 'original_question_options',
 'original_reference_response_cot',
 'original_reference_response_raw_response',
 'original_reference_response_parsed_response',
 'original_reference_response_answer',
 'original_reference_response_model_info_model',
 'original_reference_response_model_info_temperature',
 'original_reference_response_model_info_max_tokens',
 'original_reference_response_model_info_thinking',
 'original_reference_response_model_info_seed',
 'original_reference_response_model_info_additional_params',
 'original_reference_response_predictor_answers',
 'original_reference_response_predictor_names',
 'original_reference_response_input_tokens',
 'original_reference_response_reasoning_tokens',
 'original_reference_response_output_tokens',
 'counterfactual_generator_model',
 'counterfactual_generator_method',

In [11]:
import numpy as np
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

num_tokens_per_row = []
num_tokens_per_100 = []

for i in range(len(df)):
    num_tokens = len(tok.encode(df['original_reference_response_cot'].iloc[i]))
    num_tokens_per_row.append(num_tokens)

    if (i % 100 == 0):
        num_tokens_per_100.append(num_tokens)
                        
    
num_tokens_per_row = np.array(num_tokens_per_row)
mean = num_tokens_per_row.mean()
std = num_tokens_per_row.std()

int(mean), int(std), num_tokens_per_100



(1002,
 752,
 [4806,
  725,
  675,
  572,
  2874,
  528,
  816,
  633,
  1328,
  490,
  1059,
  1454,
  466,
  1090,
  802,
  774,
  2025,
  3815,
  1126,
  732,
  575,
  2031,
  1013,
  679,
  856,
  3270,
  783,
  691,
  500,
  569,
  475,
  3651,
  250,
  1027,
  1469,
  577,
  1113,
  318,
  1077,
  4098,
  899,
  1167,
  1153,
  851,
  743,
  775,
  1180,
  536,
  678,
  1020,
  475,
  2494,
  914,
  700,
  678,
  783,
  633,
  2946,
  1083,
  802,
  792,
  267,
  1050,
  918,
  1471,
  1289,
  733,
  3066,
  690,
  3961])

In [7]:
len(df)

7000

In [65]:
df.columns
# pprint(df['counterfactual_reference_response_cot'].iloc[1])
# print("\n")
# pprint(df['original_reference_response_cot'].iloc[1])
pprint(df['original_reference_response_raw_response'].iloc[1])

('[ANSWER]  \n'
 'NO  \n'
 '\n'
 '[EXPLANATION]  \n'
 'The patient’s clinical profile suggests a low likelihood of heart disease. '
 'Key factors include:  \n'
 '1. **Asymptomatic presentation**: Absence of chest pain or exercise-induced '
 'angina reduces suspicion of ischemic heart disease.  \n'
 '2. **Normal resting ECG**: A normal baseline ECG is reassuring, though flat '
 'ST segments may not be pathologic in isolation.  \n'
 '3. **Normal cholesterol and fasting blood sugar**: These are protective '
 'factors against atherosclerosis and diabetes-related cardiovascular risk.  \n'
 '4. **Elevated blood pressure**: While hypertension is a known risk factor, '
 'it does not definitively indicate heart disease without additional evidence '
 '(e.g., target organ damage, proteinuria, or left ventricular '
 'hypertrophy).  \n'
 '5. **Age**: Being 50–60 years old increases baseline risk, but this is '
 'offset by the absence of symptoms and normal laboratory findings.  \n'
 '\n'
 'The comb

In [ ]:
# need to fix data first, there are lots of None Rows
def calc_token_stats(df):
    model =  df['original_reference_response_model_info_model'].iloc[0]
    print(f"Model used to generate reference response: {model}")
    tok = AutoTokenizer.from_pretrained(f"{model}")

    num_tokens_per_row = []
    num_tokens_per_100 = []

    for i in range(len(df)):
        num_tokens = len(tok.encode(df['original_reference_response_cot'].iloc[i]))
        num_tokens_per_row.append(num_tokens)

        if (i % 100 == 0):
            num_tokens_per_100.append(num_tokens)                      
        
    num_tokens_per_row = np.array(num_tokens_per_row)
    mean = num_tokens_per_row.mean()
    std = num_tokens_per_row.std()

    return(int(mean), int(std), num_tokens_per_100, num_tokens_per_row)

df = pd.read_parquet("reference_answers.parquet")

mean,std, num_tokens_per_100,num_tokens_per_row = calc_token_stats(df)




Model used to generate reference response: Qwen/Qwen3-8B


array([ 756.  , 1925.1 , 2808.1 , 3927.24])

In [41]:
percentiles = np.percentile(num_tokens_per_row, [50, 90, 95, 99])
max = num_tokens_per_row.max()
min = num_tokens_per_row.min()

print(f"Model used is: {df['original_reference_response_model_info_model'].iloc[0]}")
percentiles,max,min

# stats for

Model used is: Qwen/Qwen3-8B


(array([ 756.  , 1925.1 , 2808.1 , 3927.24]), np.int64(7010), np.int64(193))

In [51]:
df.columns

Index(['original_dataset', 'original_question', 'original_question_prompt',
       'original_question_idx', 'original_ground_truth',
       'original_answer_first', 'original_description',
       'original_question_options', 'original_reference_response_cot',
       'original_reference_response_raw_response',
       'original_reference_response_parsed_response',
       'original_reference_response_answer',
       'original_reference_response_model_info_model',
       'original_reference_response_model_info_temperature',
       'original_reference_response_model_info_max_tokens',
       'original_reference_response_model_info_thinking',
       'original_reference_response_model_info_seed',
       'original_reference_response_model_info_additional_params',
       'original_reference_response_predictor_answers',
       'original_reference_response_predictor_names',
       'original_reference_response_input_tokens',
       'original_reference_response_reasoning_tokens',
       'original_re

In [56]:
from pprint import pprint
# objective prepend 

for idx, prompt in enumerate(df['original_question_prompt']):
    df.loc[idx,"original_question_prompt"] = prompt.replace("detailed","concise")

# pprint(df['original_question_prompt'].iloc[10])
# 'counterfactual_prompt_with_explanation',
#  'counterfactual_prompt_without_explanation

pprint(df['counterfactual_prompt_with_explanation'].iloc[0])

('You are an ethics researcher studying how people make moral judgments about '
 "autonomous vehicle dilemmas. Your task is to study an evaluator's assessment "
 'of a reference scenario and predict how the evaluator would judge a new, '
 "counterfactual scenario. The evaluator's reasoning may differ from your own "
 'beliefs, but your aim is to predict their behavior by simulating their moral '
 'reasoning.\n'
 '\n'
 'You will be shown:\n'
 '1. A "reference scenario" with another evaluator\'s judgment and their '
 'ethical reasoning\n'
 '2. A "counterfactual scenario" with different characteristics\n'
 '\n'
 "Your Task: Based on the evaluator's judgment and reasoning for the reference "
 "scenario, predict what you think the evaluator's judgment of the "
 'counterfactual scenario would be. This may differ from your own judgment. '
 "Follow the evaluator's explanation and reasoning to predict how they will "
 'judge the new scenario.\n'
 '\n'
 '--- REFERENCE SCENARIO ---\n'
 'Consider 

In [ ]:
# comparing flops

# A40 : 149.7 | 299.4*
# L40s ; 362 vs 181 TFLOPS dense

# VRAM capacity (measured in GB) — how much data fits. 48GB, 64GB, 96GB. This determines whether your model + KV cache fit and how big a batch you can hold.
# Memory bandwidth (measured in GB/s or TB/s) — how fast data moves between the VRAM and the compute cores. 864 GB/s, 1.79 TB/s. This is a speed, not a capacity.
# Compute / TFLOPS — how fast the cores do math once the data arrives.

